In [1]:
import arcpy
import os
import datetime
import sqlite3
import pandas as pd
from arcpy.sa import *

# Real-time logging function
def log_step(message):
    current_time = datetime.datetime.now().strftime("%H:%M:%S")
    arcpy.AddMessage(f"[{current_time}] {message}")
    print(f"[{current_time}] {message}")

In [2]:
log_step("Querying raw SQLite database for source validation...")

# Database paths
sqlite_db_path = r"C:\Users\<YOUR_USERNAME>\SSURGO Portal\SQLite_Template_sqlite\SQLite_Template.sqlite"
sqlite_table_name = "rating_DrainClass_DCD" 

# Connect to the raw source
conn = sqlite3.connect(sqlite_db_path)

# Write a pure SQL query to count the rows natively in the database
validation_query = f"""
SELECT DrainClass_DCD, COUNT(*) as Source_Count
FROM {sqlite_table_name}
GROUP BY DrainClass_DCD
ORDER BY Source_Count DESC
"""

# Fetch and print the results
source_summary = pd.read_sql_query(validation_query, conn)
conn.close()

print("\n--- RAW SQLITE SOURCE COUNTS ---")
print(source_summary.to_string(index=False))
print("--------------------------------\n")

[18:17:10] Querying raw SQLite database for source validation...

--- RAW SQLITE SOURCE COUNTS ---
              DrainClass_DCD  Source_Count
                Well drained          9534
     Moderately well drained          1673
                        None           772
              Poorly drained           755
Somewhat excessively drained           637
     Somewhat poorly drained           568
         Very poorly drained           441
         Excessively drained           268
                  Subaqueous            49
--------------------------------



In [3]:
log_step("--- STARTING SSURGO RAW EXTRACTION (GDB IN-MEMORY METHOD) ---")

# --- 1. Define Explicit Paths ---
base_dir = r"C:\Users\<YOUR_USERNAME>\Documents\ArcGIS\Projects\SOIL_04272026"
baked_raster_path = os.path.join(base_dir, "Baked_Drainage_10m.tif")

raw_raster = r"C:\Users\<YOUR_USERNAME>\SSURGO Portal\SQLite_Template_sqlite\muraster_10m.tif"

# SQLite Paths
sqlite_db_path = r"C:\Users\<YOUR_USERNAME>\SSURGO Portal\SQLite_Template_sqlite\SQLite_Template.sqlite"
sqlite_table_name = "rating_DrainClass_DCD" 

# Temp paths and variables
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
temp_csv_path = os.path.join(base_dir, f"cached_ssurgo_{timestamp}.csv")
scratch_gdb = arcpy.env.scratchGDB
gdb_table_name = f"ssurgo_tbl_{timestamp}"
gdb_table_path = os.path.join(scratch_gdb, gdb_table_name)
raster_layer = f"muraster_lyr_{timestamp}"

# --- 2. Global Environment Settings ---
arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")
arcpy.env.parallelProcessingFactor = "0"

# --- 3. Execution Steps ---
try:
    log_step("Step A: Extracting SQLite directly to CSV and applying Drainage mapping...")
    conn = sqlite3.connect(sqlite_db_path)
    query = f"SELECT * FROM {sqlite_table_name}"
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # --- NEW ADDITION: Map text to physical drainage scale ---
    drainage_dict = {
        "Excessively drained": 1,
        "Somewhat excessively drained": 2,
        "Well drained": 3,
        "Moderately well drained": 4,
        "Somewhat poorly drained": 5,
        "Poorly drained": 6,
        "Very poorly drained": 7,
        "Subaqueous": 8
    }
    
    # Create a new numeric column based on the text column
    df['Drainage_Rank'] = df['DrainClass_DCD'].map(drainage_dict).fillna(0).astype(int)
    # ---------------------------------------------------------

    log_step("Verifying Drainage Reclass Results...")
    print("\n--- RECLASS SUMMARY ---")
    
    # 1. Get counts and drop NaNs (dropna=True is the default, but good to be explicit)
    summary = df[['DrainClass_DCD', 'Drainage_Rank']].value_counts(dropna=True).reset_index(name='Count')
    
    # 2. Sort the table numerically by your new Drainage Rank (1 through 8)
    summary = summary.sort_values(by='Drainage_Rank')
    
    # 3. Print cleanly without the index numbers
    print(summary.to_string(index=False))
    print("-----------------------\n")
    # --------------------------------------

    df.to_csv(temp_csv_path, index=False)
    
    log_step(f"Step B: Importing CSV into native Geodatabase ({gdb_table_name})...")
    arcpy.conversion.TableToTable(temp_csv_path, scratch_gdb, gdb_table_name)

    log_step("Step C: Creating a virtual raster layer and applying in-memory join...")
    
    # SAFETY CHECK: Verify ArcGIS can actually see the raster before running
    if not arcpy.Exists(raw_raster):
        raise FileNotFoundError(f"ArcGIS STILL cannot find the raster at this path:\n{raw_raster}")
        
    arcpy.management.MakeRasterLayer(raw_raster, raster_layer)
    
    arcpy.management.AddJoin(
        in_layer_or_view=raster_layer,
        in_field="Value",
        join_table=gdb_table_path,
        join_field="mukey"
    )

    log_step("Step D: Running Lookup to create standalone drainage raster...")
    
    # --- NEW ADDITION: Target the newly created numeric column for the raster values ---
    joined_field_name = f"{gdb_table_name}.Drainage_Rank"
    # ---------------------------------------------------------------------------------
    
    drainage_raster = Lookup(raster_layer, joined_field_name)
    drainage_raster.save(baked_raster_path)

    log_step(f"✅ SUCCESS! Join baked successfully into: {baked_raster_path}")

except Exception as e:
    log_step(f"❌ ERROR ENCOUNTERED: {str(e)}")

finally:
    # --- 4. Cleanup ---
    log_step("Step E: Cleaning up temporary files to save disk space...")
    try:
        arcpy.management.RemoveJoin(raster_layer)
    except:
        pass
    if os.path.exists(temp_csv_path):
        try:
            os.remove(temp_csv_path)
        except:
            pass
    if arcpy.Exists(gdb_table_path):
        arcpy.management.Delete(gdb_table_path)
        
    log_step("--- SSURGO EXTRACTION COMPLETE ---")

[18:17:10] --- STARTING SSURGO RAW EXTRACTION (GDB IN-MEMORY METHOD) ---
[18:17:10] Step A: Extracting SQLite directly to CSV and applying Drainage mapping...
[18:17:10] Verifying Drainage Reclass Results...

--- RECLASS SUMMARY ---
              DrainClass_DCD  Drainage_Rank  Count
         Excessively drained              1    268
Somewhat excessively drained              2    637
                Well drained              3   9534
     Moderately well drained              4   1673
     Somewhat poorly drained              5    568
              Poorly drained              6    755
         Very poorly drained              7    441
                  Subaqueous              8     49
-----------------------

[18:17:10] Step B: Importing CSV into native Geodatabase (ssurgo_tbl_20260506_181710)...
[18:17:11] Step C: Creating a virtual raster layer and applying in-memory join...
[18:17:12] Step D: Running Lookup to create standalone drainage raster...
[18:20:33] ✅ SUCCESS! Join baked succe

In [4]:
log_step("Checking metadata of baked raster for soil drainage")

if baked_raster_path:
    desc = arcpy.Describe(baked_raster_path)     
    sr = desc.spatialReference
    
    print(f"File: {drainage_raster}")
    print(f"  X Cell Size: {desc.meanCellWidth}")
    print(f"  Y Cell Size: {desc.meanCellHeight}")

    # --- NEW METADATA PROPERTIES ---
    print(f"  Coordinate System: {sr.name}")
    print(f"  Linear Unit: {sr.linearUnitName}") # E.g., 'Meter' or 'Foot_US'
    
    print(f"  Pixel Type: {desc.pixelType}")     # E.g., 'F32' (32-bit float) or 'U8' (8-bit integer)
    print(f"  Band Count: {desc.bandCount}")
    print(f"  NoData Value: {desc.noDataValue}") 
    # -------------------------------
    
    print("-" * 50)
else:
    log_step("No drainage raster found in the folder.")

[18:20:33] Checking metadata of baked raster for soil drainage
File: C:\Users\<YOUR_USERNAME>\Documents\ArcGIS\Projects\SOIL_04272026\Baked_Drainage_10m.tif
  X Cell Size: 10.0
  Y Cell Size: 10.0
  Coordinate System: NAD_1983_Contiguous_USA_Albers
  Linear Unit: Meter
  Pixel Type: S8
  Band Count: 1
  NoData Value: 15
--------------------------------------------------


In [5]:
log_step("--- STARTING PROJECTION PHASE ---")

# --- 1. Define Paths ---
base_dir = r"C:\Users\<YOUR_USERNAME>\Documents\ArcGIS\Projects\SOIL_04272026"
baked_raster_path = os.path.join(base_dir, "Baked_Drainage_10m.tif")
proj_raster_path = os.path.join(base_dir, "Proj_Drainage_10m.tif")

# --- 2. Environment Settings ---
arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

# Target CRS: NAD 1983 StatePlane North Carolina FIPS 3200
out_crs = arcpy.SpatialReference(32119)
arcpy.env.outputCoordinateSystem = out_crs

try:
    log_step(f"Projecting Albers soil raster to CRS 32119...")
    arcpy.management.ProjectRaster(
        in_raster=baked_raster_path, 
        out_raster=proj_raster_path, 
        out_coor_system=out_crs, 
        resampling_type="NEAREST", # CRITICAL: Nearest Neighbor for categorical data
        cell_size="10 10"
    )
    log_step(f"✅ Projection successfully saved as: Proj_Drainage_10m.tif")
except Exception as e:
    log_step(f"❌ ERROR: {str(e)}")

[18:20:33] --- STARTING PROJECTION PHASE ---
[18:20:33] Projecting Albers soil raster to CRS 32119...
[18:23:49] ✅ Projection successfully saved as: Proj_Drainage_10m.tif


In [6]:
log_step("Checking metadata of projected soil drainage raster")

if proj_raster_path:
    desc = arcpy.Describe(proj_raster_path)     
    sr = desc.spatialReference
    
    print(f"File: {proj_raster_path}")
    print(f"  X Cell Size: {desc.meanCellWidth}")
    print(f"  Y Cell Size: {desc.meanCellHeight}")

    # --- NEW METADATA PROPERTIES ---
    print(f"  Coordinate System: {sr.name}")
    print(f"  Linear Unit: {sr.linearUnitName}") # E.g., 'Meter' or 'Foot_US'
    
    print(f"  Pixel Type: {desc.pixelType}")     # E.g., 'F32' (32-bit float) or 'U8' (8-bit integer)
    print(f"  Band Count: {desc.bandCount}")
    print(f"  NoData Value: {desc.noDataValue}") 
    # -------------------------------
    
    print("-" * 50)
else:
    log_step("No projected drainage raster found in the folder.")

[18:23:49] Checking metadata of projected soil drainage raster
File: C:\Users\<YOUR_USERNAME>\Documents\ArcGIS\Projects\SOIL_04272026\Proj_Drainage_10m.tif
  X Cell Size: 10.0
  Y Cell Size: 10.0
  Coordinate System: NAD_1983_StatePlane_North_Carolina_FIPS_3200
  Linear Unit: Meter
  Pixel Type: S8
  Band Count: 1
  NoData Value: 15
--------------------------------------------------


In [7]:
log_step("--- STARTING RESAMPLE & CLIP TO TEMPLATE ---")

# --- 1. Define Paths ---
snap_raster_path = os.path.join(base_dir, "Reference", "apnep-wsi-conserve-wetlands.tif")
final_soil_30m_path = os.path.join(base_dir, "APNEP_Drainage_30m.tif")

# Create a temporary path for the resampled raster to avoid the in-memory bug
temp_resampled_path = os.path.join(arcpy.env.scratchGDB, "temp_resampled_30m")

snap_desc = arcpy.Describe(snap_raster_path)

# --- 2. Lock Global Environments to Template ---
arcpy.env.snapRaster = snap_raster_path
arcpy.env.cellSize = snap_raster_path
arcpy.env.outputCoordinateSystem = out_crs
arcpy.env.extent = snap_desc.extent
arcpy.env.mask = snap_raster_path # Critical for the extraction step
arcpy.env.overwriteOutput = True

try:
    # 3. Resample (Scales 10x10 up to 30x30 using Majority rule)
    log_step("Resampling to 30m using Majority rule...")
    
    # We can use proj_raster_path directly now because it is already an Integer!
    arcpy.management.Resample(
        in_raster=proj_raster_path,
        out_raster=temp_resampled_path,
        cell_size=30,
        resampling_type="MAJORITY" 
    )

    # 4. Extract by Mask (Crops to exact template dimensions)
    log_step("Clipping shape to match WSI template...")
    final_soil_30m = ExtractByMask(
        in_raster=temp_resampled_path,
        in_mask_data=snap_raster_path
    )

    # 5. Save the pure integer raster
    final_soil_30m.save(final_soil_30m_path)
    
    # --- 6. ADD STRING REFERENCE BACK TO FINAL RASTER ---
    log_step("Building Raster Attribute Table and attaching text labels...")
    
    arcpy.management.BuildRasterAttributeTable(final_soil_30m_path)
    
    arcpy.management.AddField(
        in_table=final_soil_30m_path, 
        field_name="DrainClass", 
        field_type="TEXT", 
        field_length=50
    )
    
    code_block = """def get_name(val):
    mapping = {
        1: 'Excessively drained',
        2: 'Somewhat excessively drained',
        3: 'Well drained',
        4: 'Moderately well drained',
        5: 'Somewhat poorly drained',
        6: 'Poorly drained',
        7: 'Very poorly drained',
        8: 'Subaqueous',
        0: 'Unmapped/Water'
    }
    return mapping.get(val, 'Unknown')"""
    
    arcpy.management.CalculateField(
        in_table=final_soil_30m_path, 
        field="DrainClass", 
        expression="get_name(!Value!)", 
        expression_type="PYTHON3", 
        code_block=code_block
    )

    # Validation check
    final_desc = arcpy.Describe(final_soil_30m_path)
    log_step(f"✅ SUCCESS! Final Output Shape: {final_desc.width} x {final_desc.height}")

    # --- 7. PRINT SUMMARY OF FINAL ATTRIBUTE TABLE ---
    log_step("Verifying Final 30m Raster Attributes...")
    print("\n--- FINAL 30m RASTER SUMMARY ---")
    print(f"{'Value':<10} | {'Count (Pixels)':<15} | {'DrainClass'}")
    print("-" * 55)
    
    with arcpy.da.SearchCursor(final_soil_30m_path, ["Value", "Count", "DrainClass"]) as cursor:
        for row in cursor:
            print(f"{row[0]:<10} | {row[1]:<15} | {row[2]}")
    print("--------------------------------\n")

except Exception as e:
    log_step(f"❌ ERROR: {str(e)}")

finally:
    # 8. Clear environments
    arcpy.env.snapRaster = None
    arcpy.env.cellSize = None
    arcpy.env.outputCoordinateSystem = None
    arcpy.env.extent = None
    arcpy.env.mask = None
    
    # Clean up the temporary resampled file
    if arcpy.Exists(temp_resampled_path):
        arcpy.management.Delete(temp_resampled_path)
        
    log_step("--- SSURGO WORKFLOW COMPLETE ---")

[18:23:49] --- STARTING RESAMPLE & CLIP TO TEMPLATE ---
[18:23:49] Resampling to 30m using Majority rule...
[18:23:56] Clipping shape to match WSI template...
[18:24:04] Building Raster Attribute Table and attaching text labels...
[18:24:05] ✅ SUCCESS! Final Output Shape: 11378 x 9812
[18:24:05] Verifying Final 30m Raster Attributes...

--- FINAL 30m RASTER SUMMARY ---
Value      | Count (Pixels)  | DrainClass
-------------------------------------------------------
1          | 561612.0        | Excessively drained
2          | 534683.0        | Somewhat excessively drained
3          | 17684939.0      | Well drained
4          | 9581195.0       | Moderately well drained
5          | 4247808.0       | Somewhat poorly drained
6          | 12261745.0      | Poorly drained
7          | 10916549.0      | Very poorly drained
8          | 179237.0        | Subaqueous
--------------------------------

[18:24:06] --- SSURGO WORKFLOW COMPLETE ---


In [8]:
log_step("Checking metadata of clipped soil drainage raster")

if final_soil_30m_path:
    desc = arcpy.Describe(final_soil_30m_path)     
    sr = desc.spatialReference
    
    print(f"File: {final_soil_30m_path}")
    print(f"  X Cell Size: {desc.meanCellWidth}")
    print(f"  Y Cell Size: {desc.meanCellHeight}")

    # --- NEW METADATA PROPERTIES ---
    print(f"  Coordinate System: {sr.name}")
    print(f"  Linear Unit: {sr.linearUnitName}") # E.g., 'Meter' or 'Foot_US'
    
    print(f"  Pixel Type: {desc.pixelType}")     # E.g., 'F32' (32-bit float) or 'U8' (8-bit integer)
    print(f"  Band Count: {desc.bandCount}")
    print(f"  NoData Value: {desc.noDataValue}") 
    print(f"  Output Shape: {desc.width} x {desc.height}")
    # -------------------------------
    
    print("-" * 50)
else:
    log_step("No projected drainage raster found in the folder.")

[18:24:06] Checking metadata of clipped soil drainage raster
File: C:\Users\<YOUR_USERNAME>\Documents\ArcGIS\Projects\SOIL_04272026\APNEP_Drainage_30m.tif
  X Cell Size: 30.0
  Y Cell Size: 30.0
  Coordinate System: NAD_1983_StatePlane_North_Carolina_FIPS_3200
  Linear Unit: Meter
  Pixel Type: S8
  Band Count: 1
  NoData Value: 15
  Output Shape: 11378 x 9812
--------------------------------------------------


## Value conversion

In [9]:
import arcpy
import os
from arcpy.sa import *

# --- 1. Define Paths ---
base_dir = r"C:\Users\<YOUR_USERNAME>\Documents\ArcGIS\Projects\SOIL_04272026"
final_soil_30m_path = os.path.join(base_dir, "APNEP_Drainage_30m.tif")
wsi_score_raster_path = os.path.join(base_dir, "apnep-wsi-soildrainage.tif")

# --- 2. Environment Settings ---
arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

try:
    print("Step 1: Adding WSI_Score field to the raster attribute table...")
    # Add a standard Float field
    arcpy.management.AddField(
        in_table=final_soil_30m_path, 
        field_name="WSI_Score", 
        field_type="FLOAT"
    )

    print("Step 2: Calculating WSI Suitability Scores (using -1.0 as a dummy NoData)...")
    # Map the existing 1-8 integers. Use -1.0 for areas that will be carved out.
    code_block = """def get_wsi_score(val):
    mapping = {
        1: 0.0,   # Excessively drained
        2: 0.0,   # Somewhat excessively drained
        3: 0.0,   # Well drained
        4: 0.5,   # Moderately well drained
        5: 0.5,   # Somewhat poorly drained
        6: 1.0,   # Poorly drained
        7: 1.0,   # Very poorly drained
        8: -1.0,  # Subaqueous -> Will become NoData
        0: -1.0   # Unmapped/Water -> Will become NoData
    }
    # Default to -1.0 just in case
    return mapping.get(val, -1.0)"""

    arcpy.management.CalculateField(
        in_table=final_soil_30m_path, 
        field="WSI_Score", 
        expression="get_wsi_score(!Value!)", 
        expression_type="PYTHON3", 
        code_block=code_block
    )

    print("Step 3: Extracting scores and carving out NoData...")
    # A. Create the initial Float raster with the -1.0 values
    raw_wsi = Lookup(final_soil_30m_path, "WSI_Score")
    
    # B. Use SetNull to convert those -1.0 values into true, transparent NoData
    final_wsi = SetNull(raw_wsi == -1.0, raw_wsi)
    
    # C. Save the final perfected output
    final_wsi.save(wsi_score_raster_path)

    print(f"✅ SUCCESS! WSI Drainage Score raster saved to:\n   {wsi_score_raster_path}")

except Exception as e:
    print(f"❌ ERROR: {str(e)}")

Step 1: Adding WSI_Score field to the raster attribute table...
Step 2: Calculating WSI Suitability Scores (using -1.0 as a dummy NoData)...
Step 3: Extracting scores and carving out NoData...
✅ SUCCESS! WSI Drainage Score raster saved to:
   C:\Users\<YOUR_USERNAME>\Documents\ArcGIS\Projects\SOIL_04272026\apnep-wsi-soildrainage.tif


In [10]:
import numpy as np
log_step("Validating soil drainage suitability...")

if arcpy.Exists(wsi_score_raster_path):
    desc = arcpy.Describe(wsi_score_raster_path)
    print(f"File: {wsi_score_raster_path}")
    print(f"  Cell Size: {desc.meanCellWidth} x {desc.meanCellHeight}")
    print(f"  CRS: {desc.spatialReference.name}")
    print(f"  Pixel Type: {desc.pixelType}")
    print(f"  Dimensions: {desc.width} x {desc.height}")
    print(f"  NoData: {desc.noDataValue}")

    # Convert to array and handle NoData
    # Using np.nan allows for clean filtering of NoData (Subaqueous/Water)
    soil_arr = arcpy.RasterToNumPyArray(wsi_score_raster_path, nodata_to_value=np.nan)
    valid = soil_arr[~np.isnan(soil_arr)]
    total = int(valid.size)

    # Labels based on the logic in ssurgo_soil_processing.ipynb
    labels = {
        1.0: "Poorly/Very Poorly",
        0.5: "Mod/Somewhat Poorly",
        0.0: "Well/Excessive"
    }

    print(f"\n{'Suitability':<14}{'Drainage Group':<20}{'Pixels':>15}{'%':>8}")
    print("-" * 60)
    
    # Iterate through the specific WSI scores defined in the processing script[cite: 1]
    for score in [1.0, 0.5, 0.0]:
        cnt = int(np.isclose(valid, score).sum())
        pct = (100.0 * cnt / total) if total > 0 else 0
        print(f"{score:<14.2f}{labels[score]:<20}{cnt:>15,}{pct:>7.1f}%")
        
    print(f"{'Total':<34}{total:>15,}{'100.0%':>8}")

    del soil_arr, valid
    log_step("Soil drainage validation complete.")
else:
    log_step(f"❌ ERROR: Output file not found at {wsi_score_raster_path}")

[18:24:18] Validating soil drainage suitability...
File: C:\Users\<YOUR_USERNAME>\Documents\ArcGIS\Projects\SOIL_04272026\apnep-wsi-soildrainage.tif
  Cell Size: 30.0 x 30.0
  CRS: NAD_1983_StatePlane_North_Carolina_FIPS_3200
  Pixel Type: F32
  Dimensions: 11378 x 9812
  NoData: -3.4028234663852886e+38

Suitability   Drainage Group               Pixels       %
------------------------------------------------------------
1.00          Poorly/Very Poorly       23,178,294   41.5%
0.50          Mod/Somewhat Poorly      13,829,003   24.8%
0.00          Well/Excessive           18,781,234   33.7%
Total                                  55,788,531  100.0%
[18:24:22] Soil drainage validation complete.
